In [313]:
print("DAY-2 Workshop on AI-Driven RNA Therapeutics: From Data to Drug Design")

DAY-2 Workshop on AI-Driven RNA Therapeutics: From Data to Drug Design


## feature Engineering for RNA sequence

In [314]:
from Bio.Seq import Seq
from  Bio.SeqUtils import gc_fraction

In [315]:
import pandas as pd
data={
  "sequence":[("AUCGAGCGC"),("GGCAAAAA"),("CAGGCA"),("AUGCCA")],
 
  "efficacy":[.7,.8,.2,.4]
}
df=pd.DataFrame(data)
df

,sequence,efficacy
0,AUCGAGCGC,0.7
1,GGCAAAAA,0.8
2,CAGGCA,0.2
3,AUGCCA,0.4


In [316]:
import numpy as np
mapping={
  "A":[1,0,0,0],
  "C":[0,1,0,0],
  "G":[0,0,1,0],
  "U":[0,0,0,1]
}
def one_hot_encoding(seq):
  return np.array([mapping[n]for n in seq] ).flatten()

In [317]:
df["one_hot_sequence "]=df["sequence"].apply(one_hot_encoding)
df

,sequence,efficacy,one_hot_sequence
0,AUCGAGCGC,0.7,"[1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, ..."
1,GGCAAAAA,0.8,"[0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, ..."
2,CAGGCA,0.2,"[0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, ..."
3,AUGCCA,0.4,"[1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, ..."


In [318]:
def gc_content(sequence):
    seq = Seq(sequence)
    return gc_fraction(seq) * 100

In [319]:
def gc_skew(sequence):
    seq = Seq(sequence)
    g = seq.count("G")
    c = seq.count("C")
    return (g - c) / (g + c) if (g + c) > 0 else 0

def au_skew(sequence):
    seq = Seq(sequence)
    a = seq.count("A")
    u = seq.count("U") or seq.count("T")  # handles both RNA and DNA
    return (a - u) / (a + u) if (a + u) > 0 else 0

In [320]:
df["gc_content "]=df["sequence"].apply(gc_content)
df["gc_skew"]=df["sequence"].apply(gc_skew)
df["au_skew"]=df["sequence"].apply(au_skew)
df

,sequence,efficacy,one_hot_sequence,gc_content,gc_skew,au_skew
0,AUCGAGCGC,0.7,"[1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, ...",66.666667,0.000000,0.333333
1,GGCAAAAA,0.8,"[0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, ...",37.500000,0.333333,1.000000
2,CAGGCA,0.2,"[0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, ...",66.666667,0.000000,1.000000
3,AUGCCA,0.4,"[1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, ...",50.000000,-0.333333,0.333333


## siRNA Efficiency Prediction Pipeline

In [321]:
hu=pd.read_csv("./Data/Hu.csv")
hu.head()

,siRNA,mRNA,label,y,td
0,CUAAUAUGUUAAUUGAUUU,AAAUUGAAAGGAAUUGUAUAAAUCAAUUAACAUAUUAGCUGAGUUG...,0.344519,0,"-1.6,-2.08,-10.48,0,0,-149.37999999999997,0.52..."
1,AAUAUGUUAAUUGAUUUAU,UAAAAUUGAAAGGAAUUGUAUAAAUCAAUUAACAUAUUAGCUGAGU...,0.286353,0,"0.17,-0.93,-6.82,0,0,-144.55999999999997,0.526..."
2,GAUUUAUACAAUUCCUUUC,CUUAUUUUUAGAUAAAAUUGAAAGGAAUUGUAUAAAUCAAUUAACA...,0.383296,0,"0.0,-2.35,-12.44,0,1,-163.86,0.473684210526315..."
3,CAAUUCCUUUCAAUUUUAU,UUGGUCUGCUUAUUUUUAGAUAAAAUUGAAAGGAAUUGUAUAAAUC...,0.271439,0,"-1.46,-2.11,-10.44,0,0,-152.68999999999994,0.5..."
4,CAGACCAAAAUUAAAUAAG,UGGAAUCUUAUGUAACUUUCUUAUUUAAUUUUGGUCUGCUUAUUUU...,0.389262,0,"-0.03,-2.11,-10.44,0,0,-157.33999999999997,0.1..."


In [322]:
import pandas as pd 
from collections import Counter

In [349]:
#load the data
def load_data(path):
  df=pd.read_csv(path)
  return df
  

In [324]:
#load the data
patent_ENsiRNA=load_data("./Data/patent_ENsiRNA.csv")
patent_ENsiRNA.head()


,siRNA,anti seq,sense seq,mRNA_seq,position,efficacy
0,AD-59453,AACAACAGAGATTTGCCTG,CAGGCAAATCTCTGTTGTT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,401,0.888
1,AD-59395,TTTCTCATCAATTTTTTTC,GAAAAAAATTGATGAGAAA,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,948,0.873
2,AD-59477,AAGAGTGCGGCATCTTTCC,GGAAAGATGCCGCACTCTT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,1241,0.855
3,AD-59492,TCTTGAAGAAGATGAGACA,TGTCTCATCTTCTTCAAGA,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,881,0.852
4,AD-59361,ATTGCTTGCACGTAGATGT,ACATCTACGTGCAAGCAAT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,1991,0.849


In [325]:
#drop siRNA because it just index
patent_ENsiRNA.drop("siRNA",axis=1,inplace=True)
patent_ENsiRNA.head()

,anti seq,sense seq,mRNA_seq,position,efficacy
0,AACAACAGAGATTTGCCTG,CAGGCAAATCTCTGTTGTT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,401,0.888
1,TTTCTCATCAATTTTTTTC,GAAAAAAATTGATGAGAAA,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,948,0.873
2,AAGAGTGCGGCATCTTTCC,GGAAAGATGCCGCACTCTT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,1241,0.855
3,TCTTGAAGAAGATGAGACA,TGTCTCATCTTCTTCAAGA,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,881,0.852
4,ATTGCTTGCACGTAGATGT,ACATCTACGTGCAAGCAAT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,1991,0.849


In [326]:
patent_ENsiRNA.rename(columns={
    'anti seq': 'anti_seq',
    'sense seq': 'sense_seq',
    'mRNA_seq': 'mRNA_seq'          # already fine, but keep for clarity
}, inplace=True)

# Convert sequence columns to uppercase
seq_cols = ['anti_seq', 'sense_seq', 'mRNA_seq']
patent_ENsiRNA[seq_cols] = patent_ENsiRNA[seq_cols].apply(lambda x: x.str.upper())


In [327]:
patent_ENsiRNA.head()

,anti_seq,sense_seq,mRNA_seq,position,efficacy
0,AACAACAGAGATTTGCCTG,CAGGCAAATCTCTGTTGTT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,401,0.888
1,TTTCTCATCAATTTTTTTC,GAAAAAAATTGATGAGAAA,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,948,0.873
2,AAGAGTGCGGCATCTTTCC,GGAAAGATGCCGCACTCTT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,1241,0.855
3,TCTTGAAGAAGATGAGACA,TGTCTCATCTTCTTCAAGA,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,881,0.852
4,ATTGCTTGCACGTAGATGT,ACATCTACGTGCAAGCAAT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,1991,0.849


In [328]:
df=patent_ENsiRNA

## Feature engineering


In [329]:
mapping={
  "A":[1,0,0,0],
  "C":[0,1,0,0],
  "G":[0,0,1,0],
  "U":[0,0,0,1],
  "T":[0,0,0,1]
}
def one_hot_encoding(seq):
  return np.array([mapping[n]for n in seq] ).flatten()

In [330]:
def gc_content(sequence):
    seq = Seq(sequence)
    return gc_fraction(seq) * 100

def gc_skew(sequence):
    seq = Seq(sequence)
    g = seq.count("G")
    c = seq.count("C")
    return (g - c) / (g + c) if (g + c) > 0 else 0

def au_skew(sequence):
    seq = Seq(sequence)
    a = seq.count("A")
    u = seq.count("U") or seq.count("T")  # handles both RNA and DNA
    return (a - u) / (a + u) if (a + u) > 0 else 0  

In [331]:
patent_ENsiRNA.head()

,anti_seq,sense_seq,mRNA_seq,position,efficacy
0,AACAACAGAGATTTGCCTG,CAGGCAAATCTCTGTTGTT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,401,0.888
1,TTTCTCATCAATTTTTTTC,GAAAAAAATTGATGAGAAA,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,948,0.873
2,AAGAGTGCGGCATCTTTCC,GGAAAGATGCCGCACTCTT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,1241,0.855
3,TCTTGAAGAAGATGAGACA,TGTCTCATCTTCTTCAAGA,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,881,0.852
4,ATTGCTTGCACGTAGATGT,ACATCTACGTGCAAGCAAT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,1991,0.849


In [372]:
X=[]
y=patent_ENsiRNA["efficacy"].values

In [341]:
df=patent_ENsiRNA

In [373]:
for i in range(len(df)):

    anti = df.iloc[i]["anti_seq"]
    sense = df.iloc[i]["sense_seq"]
    mrna = df.iloc[i]["mRNA_seq"]
    pos = df.iloc[i]["position"]

    features = []

    # One-hot Encoding

    mapping = {'A':[1,0,0,0], 'U':[0,1,0,0], 'G':[0,0,1,0], 'C':[0,0,0,1]}
    for base in anti:
        features.extend(mapping.get(base, [0,0,0,0]))

    # GC Content in Antisense

    gc = (anti.count('G') + anti.count('C')) / len(anti)
    features.append(gc)

    # NT Count

    features.append(anti.count('A'))
    features.append(anti.count('U'))
    features.append(anti.count('G'))
    features.append(anti.count('C'))

    # Dinucleotide Count

    pairs = [anti[j:j+2] for j in range(len(anti)-1)]
    count = Counter(pairs)
    dinucs = [a+b for a in "AUGC" for b in "AUGC"]
    for d in dinucs:
        features.append(count[d])

    # GC Content in Seed Region

    seed = anti[1:8]
    seed_gc = (seed.count('G') + seed.count('C')) / len(seed)
    features.append(seed_gc)

    # Seed Region NT Count

    features.append(seed.count('A'))
    features.append(seed.count('U'))
    features.append(seed.count('G'))
    features.append(seed.count('C'))

    # GC Content in Sense

    sense_gc = (sense.count('G') + sense.count('C')) / len(sense)
    features.append(sense_gc)

    # DUPLEX ENERGY

    energy = 0
    for a, s in zip(anti, sense):
        if (a == 'G' and s == 'C') or (a == 'C' and s == 'G'):
            energy -= 2
        elif (a == 'A' and s == 'U') or (a == 'U' and s == 'A'):
            energy -= 1
    features.append(energy)

    # End Stability

    five = anti[:4]
    three = anti[-4:]
    gc5 = (five.count('G') + five.count('C')) / len(five)
    gc3 = (three.count('G') + three.count('C')) / len(three)
    features.append(gc5 - gc3)

    # GC Content in mRNA

    mrna_gc = (mrna.count('G') + mrna.count('C')) / len(mrna)
    features.append(mrna_gc)
    features.append(len(mrna))

    # Antisense-mRNA Interaction Feature

    matches = 0
    mismatches = 0
    for a, m in zip(anti, mrna[:len(anti)]):
        if (a == 'A' and m == 'U') or \
           (a == 'U' and m == 'A') or \
           (a == 'G' and m == 'C') or \
           (a == 'C' and m == 'G'):
            matches += 1
        else:
            mismatches += 1

    features.append(matches / len(anti))
    features.append(mismatches)

    # Seed matching

    seed = anti[1:8]
    target = mrna[:7]
    seed_match = 0
    for a, m in zip(seed, target):
        if (a == 'A' and m == 'U') or \
           (a == 'U' and m == 'A') or \
           (a == 'G' and m == 'C') or \
           (a == 'C' and m == 'G'):
            seed_match += 1
    features.append(seed_match)

    features.append(pos)
    features.append(np.log1p(pos))

    X.append(features)

In [374]:
X = np.array(X)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (232, 112)
Target shape: (232,)


In [375]:
X[0]

array([ 1.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        1.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  1.00000000e+00,
        1.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        1.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  1.00000000e+00,
        1.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  1.00000000e+00,  0.00000000e+00,
        1.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  1.00000000e+00,  0.00000000e+00,
        1.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  

In [377]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Hu data

In [370]:
import pandas as pd
import numpy as np
from collections import Counter

# Assuming load_data returns a DataFrame
Hu = load_data("./Data/Hu.csv")
# (Hu.head())
(Hu.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2361 entries, 0 to 2360
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   siRNA   2361 non-null   object 
 1   mRNA    2361 non-null   object 
 2   label   2361 non-null   float64
 3   y       2361 non-null   int64  
 4   td      2361 non-null   object 
dtypes: float64(1), int64(1), object(3)
memory usage: 92.4+ KB
